In [ ]:
import os
import uuid

def clean_and_rename_dataset(images_dir, masks_dir, prefix):
    print(f"Scanning directories:\nImages: {images_dir}\nMasks: {masks_dir}\n")

    image_files = [f for f in os.listdir(images_dir) if os.path.isfile(os.path.join(images_dir, f))]
    mask_files = [f for f in os.listdir(masks_dir) if os.path.isfile(os.path.join(masks_dir, f))]

    image_map = {os.path.splitext(f)[0]: f for f in image_files}
    mask_map = {os.path.splitext(f)[0]: f for f in mask_files}

    image_bases = set(image_map.keys())
    mask_bases = set(mask_map.keys())

    paired_bases = image_bases.intersection(mask_bases)
    unpaired_images = image_bases - mask_bases
    unpaired_masks = mask_bases - image_bases

    if unpaired_images:
        print(f"Deleting {len(unpaired_images)} unpaired images...")
        for base in unpaired_images:
            filepath = os.path.join(images_dir, image_map[base])
            os.remove(filepath)
            print(f"  [-] Deleted: {image_map[base]}")

    if unpaired_masks:
        print(f"\nDeleting {len(unpaired_masks)} unpaired masks...")
        for base in unpaired_masks:
            filepath = os.path.join(masks_dir, mask_map[base])
            os.remove(filepath)
            print(f"  [-] Deleted: {mask_map[base]}")


    print(f"\nPreparing to rename {len(paired_bases)} valid pairs...")
    
    sorted_bases = sorted(list(paired_bases))
    temp_mapping = []

    for base in sorted_bases:
        old_img_name = image_map[base]
        old_mask_name = mask_map[base]
        
        img_ext = os.path.splitext(old_img_name)[1].lower()
        mask_ext = os.path.splitext(old_mask_name)[1].lower()
        
        if img_ext in ['.jpeg', '.jpg', '.jpe']:
            img_ext = '.jpg'
        
        temp_base = str(uuid.uuid4())
        
        temp_img_path = os.path.join(images_dir, temp_base + img_ext)
        temp_mask_path = os.path.join(masks_dir, temp_base + mask_ext)

        os.rename(os.path.join(images_dir, old_img_name), temp_img_path)
        os.rename(os.path.join(masks_dir, old_mask_name), temp_mask_path)
        
        temp_mapping.append((temp_img_path, temp_mask_path, img_ext, mask_ext))

    print(f"Applying final names with prefix '{prefix}' and standardizing extensions...")
    
    for index, (temp_img_path, temp_mask_path, img_ext, mask_ext) in enumerate(temp_mapping, start=1):
        new_base = f"{prefix}-{index}"
        
        new_img_path = os.path.join(images_dir, new_base + img_ext)
        new_mask_path = os.path.join(masks_dir, new_base + mask_ext)
        
        # Rename from temp to final
        os.rename(temp_img_path, new_img_path)
        os.rename(temp_mask_path, new_mask_path)

    print(f"\n{len(paired_bases)} pairs have been cleanly formatted. All JPEGs are now .jpg")

In [8]:
IMAGES_DIRECTORY = "data/final/meat/JPEGImages"
MASKS_DIRECTORY = "data/final/meat/SegmentationClass"
NEW_PREFIX = "meat"
    
clean_and_rename_dataset(IMAGES_DIRECTORY, MASKS_DIRECTORY, NEW_PREFIX)

Scanning directories:
Images: data/final/meat/JPEGImages
Masks: data/final/meat/SegmentationClass


Preparing to rename 250 valid pairs...
Applying final names with prefix 'meat' and standardizing extensions...

250 pairs have been cleanly formatted. All JPEGs are now .jpg


In [9]:
IMAGES_DIRECTORY = "data/final/vegetables/JPEGImages"
MASKS_DIRECTORY = "data/final/vegetables/SegmentationClass"
NEW_PREFIX = "vegetables"
    
clean_and_rename_dataset(IMAGES_DIRECTORY, MASKS_DIRECTORY, NEW_PREFIX)

Scanning directories:
Images: data/final/vegetables/JPEGImages
Masks: data/final/vegetables/SegmentationClass


Preparing to rename 250 valid pairs...
Applying final names with prefix 'vegetables' and standardizing extensions...

250 pairs have been cleanly formatted. All JPEGs are now .jpg
